# 04 — Load model and fine-tuning

This notebook loads the model saved by notebook 03 and fine-tunes it:

1. loads `models/inceptionv3_transfer.keras`;
2. unfreezes the last **20 layers** of InceptionV3 (keeping BatchNorm frozen);
3. compiles with a small learning rate (`1e-5`);
4. trains with early stopping, checkpointing and learning-rate reduction;
5. evaluates on the test set and saves
   `models/inceptionv3_finetuned.keras`.

Notebook 03 must be run first, because this notebook starts from the
model it saves. The classifier head is kept from that model and only its
weights are refined here.

> **How to read this notebook:** every code cell is preceded by a short
> summary that explains what the step does and how to interpret its output.


## 1. Imports and paths

Imports the libraries and defines the paths of the model to load, the model to
save, and the training hyper-parameters (10 epochs, learning rate `1e-5`,
last 20 InceptionV3 layers unfrozen).

**How to read the output:** check the TensorFlow/Keras versions. If they match
the environment used for notebook 03, the saved model will load without
compatibility warnings.

In [ ]:
from pathlib import Path
import json

import cv2
import numpy as np
import tensorflow as tf
import keras
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"
MODELS_DIR = ROOT_DIR / "models"
SPLITS_PATH = PROCESSED_DIR / "splits.json"
BALANCED_PATH = PROCESSED_DIR / "train_balanced.json"
BASE_MODEL_PATH = MODELS_DIR / "inceptionv3_transfer.keras"
FINAL_MODEL_PATH = MODELS_DIR / "inceptionv3_finetuned.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 10
LEARNING_RATE = 1e-5
UNFREEZE_LAST_N = 30
AUTOTUNE = tf.data.AUTOTUNE

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)

## 2. Load the split

Reads `data/processed/splits.json` and builds the path/label arrays for the
train, validation and test subsets.

**How to read the output:** the three dataset sizes must match notebook 01.
Fine-tuning uses the **original** training split (not the oversampled one)
together with class weights, so duplicated images do not dominate training.

In [ ]:
if not SPLITS_PATH.exists():
    raise FileNotFoundError(
        f"{SPLITS_PATH} not found. Run 01_data_overview.ipynb first."
    )

with open(SPLITS_PATH) as f:
    splits = json.load(f)

classes = splits["classes"]
num_classes = len(classes)
id_to_class = {int(k): v for k, v in splits["id_to_class"].items()}
IMG_SIZE = tuple(splits.get("img_size", IMG_SIZE))

def absolute(path: str) -> str:
    return str((ROOT_DIR / path).resolve())

train_paths = [absolute(p) for p in splits["train"]["paths"]]
train_labels = np.asarray(splits["train"]["labels"], dtype=np.int32)
val_paths = [absolute(p) for p in splits["val"]["paths"]]
val_labels = np.asarray(splits["val"]["labels"], dtype=np.int32)
test_paths = [absolute(p) for p in splits["test"]["paths"]]
test_labels = np.asarray(splits["test"]["labels"], dtype=np.int32)

print("Train:", len(train_paths), "| Val:", len(val_paths), "| Test:", len(test_paths))


## 3. Build the `tf.data` pipeline

Same loader and augmentation as notebook 02, but the training set is shuffled
and augmented on the fly while validation and test stay untouched.

**How to read the output:** one batch of shape `(32, 224, 224, 3)` with pixel
values in `[0, 255]`.

In [ ]:
def load_image(path, label):
    def _load(path_tensor):
        if isinstance(path_tensor, tf.Tensor):
            path_tensor = path_tensor.numpy()
        path = path_tensor.decode("utf-8") if hasattr(path_tensor, "decode") else str(path_tensor)
        image = cv2.imread(path)
        if image is None:
            raise FileNotFoundError(f"Image not found or cannot be read: {path}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMG_SIZE)
        return image.astype(np.float32)

    image = tf.numpy_function(_load, [path], tf.float32)
    image.set_shape((IMG_SIZE[1], IMG_SIZE[0], 3))
    return image, label


def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    noise = tf.random.normal(
        shape=tf.shape(image), mean=0.0, stddev=10.0, dtype=tf.float32
    )
    image = tf.clip_by_value(image + noise, 0.0, 255.0)
    return image, label


def make_dataset(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(len(paths), seed=SEED, reshuffle_each_iteration=True)

    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)

    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)

    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset(train_paths, train_labels, training=True)
val_ds = make_dataset(val_paths, val_labels, training=False)
test_ds = make_dataset(test_paths, test_labels, training=False)

for images, labels in train_ds.take(1):
    print("Batch:", images.shape, labels.shape)


## 4. Load the model saved by notebook 03

Restores `models/inceptionv3_transfer.keras` saved by notebook 03. The model
already contains the frozen InceptionV3 backbone and the trained classifier
head in a single object, so there is nothing to rebuild here.

**How to read the output:** the model summary. The backbone is named
`inception_v3` and is still frozen at this point; the last layers (`dense`,
`dropout`, `dense_1`) are the trained head.

In [ ]:
if not BASE_MODEL_PATH.exists():
    raise FileNotFoundError(
        f"{BASE_MODEL_PATH} not found. Run 03_feature_extraction.ipynb first."
    )

model = keras.models.load_model(BASE_MODEL_PATH, compile=False)
print("Loaded:", BASE_MODEL_PATH)
model.summary()

## 5. Unfreeze the last InceptionV3 layers

Sets the backbone trainable, then freezes everything except the last
`UNFREEZE_LAST_N` layers. BatchNorm layers stay frozen even inside that block,
because updating their statistics on small batches would damage the pretrained
features.

**How to read the output:** the number of trainable backbone layers. It should
be much smaller than the total, showing that most of the network is still
frozen and only the deep layers adapt to this dataset.

In [ ]:
base_model = model.get_layer("inception_v3")
base_model.trainable = True

# Freeze everything except the last N layers.
for layer in base_model.layers[:-UNFREEZE_LAST_N]:
    layer.trainable = False

# Keep BatchNorm layers frozen during fine-tuning.
for layer in base_model.layers[-UNFREEZE_LAST_N:]:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

trainable = sum(1 for layer in base_model.layers if layer.trainable)
print(f"Trainable InceptionV3 layers: {trainable}/{len(base_model.layers)}")


## 6. Compile with a small learning rate and define callbacks

Recompiles (required after changing `trainable`) with Adam at `1e-5`, and sets
early stopping, best-model checkpointing and learning-rate reduction.

**How to read the output:** no output, only the callbacks. `1e-5` is
deliberately small: the pretrained weights must be nudged, not retrained from
scratch.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=FINAL_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.1,
        patience=3,
        min_lr=1e-7,
    ),
]


## 7. Fine-tune the model

Computes balanced class weights for the original training split and calls
`fit` on the augmented pipeline.

**How to read the output:** watch `loss` decrease and `val_loss` follow it. If
`val_loss` starts rising while training loss keeps falling, early stopping
restores the best weights and stops the run. Training can take several
minutes per epoch on CPU.

In [ ]:
# Compensate for the class imbalance in the original training split.
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_classes),
    y=train_labels,
)
class_weight = dict(enumerate(class_weights))
print("Class weights:", {id_to_class[k]: round(float(v), 3) for k, v in class_weight.items()})

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight,
)


## 8. Learning curves

Plots training/validation accuracy and loss for every epoch.

**How to read the output:** the two curves should stay close to each other. A
large gap (training much better than validation) means over-fitting; flat
curves mean the learning rate is too small or the model stopped learning.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fine-tuning — accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fine-tuning — loss")
plt.legend()

plt.tight_layout()
plt.show()


## 9. Evaluate on the test set

Computes the test loss/accuracy and a per-class precision, recall and F1
report.

**How to read the output:** compare the test accuracy with notebook 03. Look
at the per-class rows: a low recall on one class means the model misses that
diagnosis, even if the global accuracy looks good.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f"Test loss    : {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

y_true, y_pred = [], []
for images, labels in test_ds:
    probabilities = model(images, training=False)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(probabilities.numpy(), axis=1))

y_true = np.asarray(y_true)
y_pred = np.asarray(y_pred)

print(
    classification_report(
        y_true,
        y_pred,
        labels=np.arange(num_classes),
        target_names=classes,
        digits=4,
        zero_division=0,
    )
)


## 10. Confusion matrix

Cross-tabulates true and predicted classes on the test set.

**How to read the output:** the diagonal holds correct predictions. Off-diagonal
cells show which classes are confused with which; they are more informative
than a single accuracy number.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=np.arange(num_classes))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes).plot(
    xticks_rotation=45, cmap="Blues"
)
plt.title("Confusion matrix — fine-tuned InceptionV3")
plt.tight_layout()
plt.show()


## 11. Save the fine-tuned model

Writes the final model to `models/inceptionv3_finetuned.keras`.

**How to read the output:** the printed path. This file can be loaded with
`keras.models.load_model(...)` for inference or deployment.

In [ ]:
model.save(FINAL_MODEL_PATH)
print("Fine-tuned model saved:", FINAL_MODEL_PATH)


## Done

The full pipeline is now split into:

- `01_data_overview.ipynb` — data checks, EDA and split;
- `02_preprocessing_augmentation.ipynb` — preprocessing, augmentation and balancing;
- `03_feature_extraction.ipynb` — InceptionV3 features, classifier head and model saving;
- `04_fine_tuning.ipynb` — loading and fine-tuning the saved model.